# 07 — Thesis Output

**Purpose**: Generate all publication-ready tables and figures for the thesis .docx.

Collects results from notebooks 00–06 and formats them consistently.

**Outputs**: Everything in `results/thesis_figures/`

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)
figures_dir = cfg.FIGURES_DIR
figures_dir.mkdir(parents=True, exist_ok=True)

# Set consistent style
try:
    plt.style.use(cfg.PLOT_STYLE)
except Exception:
    plt.style.use('seaborn-v0_8')
matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.dpi': cfg.FIGURE_DPI,
})

## 1. Table: Market Type Inventory (Section 4.1)

In [ ]:
inv = pd.read_csv(results_dir / 'full_market_inventory.csv')

# Table 1: Market type summary
tbl1 = inv.groupby('market_type').agg(
    total=('market_id', 'size'),
    closed=('closed', 'sum'),
    resolvable=('resolvable', 'sum'),
    avg_volume=('volume_num', lambda x: f'${x.mean():,.0f}' if x.notna().any() else 'N/A'),
).sort_values('total', ascending=False)
tbl1['pct'] = (tbl1['total'] / tbl1['total'].sum() * 100).round(1)
tbl1.to_csv(figures_dir / 'table_market_types.csv')
print('Table 1: Market Type Distribution')
tbl1

In [ ]:
# Table 2: Ticker distribution
tbl2 = inv.groupby('ticker').agg(
    total=('market_id', 'size'),
    closed=('closed', 'sum'),
    resolvable=('resolvable', 'sum'),
).sort_values('total', ascending=False)
tbl2.to_csv(figures_dir / 'table_ticker_distribution.csv')
print('Table 2: Ticker Distribution')
tbl2.head(15)

## 2. Figure: Market Growth Timeline (Section 4.1)

In [ ]:
inv['start_month'] = pd.to_datetime(inv['startDate'], errors='coerce').dt.to_period('M')
monthly = inv.groupby('start_month').size()

fig, ax = plt.subplots(figsize=(10, 5))
monthly.plot.bar(ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Markets Created')
ax.set_title('Polymarket Equity Market Creation (Oct 2025 – Mar 2026)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(figures_dir / 'fig_market_growth.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
plt.show()

## 3. Calibration Figures (Sections 4.2–4.5)

In [ ]:
# Check what calibration outputs exist
cal_summary = results_dir / 'calibration_summary.csv'
if cal_summary.exists():
    cs = pd.read_csv(cal_summary)
    print('Calibration Summary:')
    print(cs.to_string(index=False))
    cs.to_csv(figures_dir / 'table_calibration_summary.csv', index=False)
else:
    print('calibration_summary.csv not found — run 03_calibration.ipynb first.')

# Copy existing calibration figures to thesis_figures
import shutil
for fig_name in ['calibration_overall.png', 'calibration_by_type.png',
                 'calibration_by_volume.png', 'calibration_by_ticker.png']:
    src = figures_dir / fig_name
    if src.exists():
        print(f'  Found: {fig_name}')
    else:
        print(f'  Missing: {fig_name}')

## 4. Event Study Tables (Section 4.6)

In [ ]:
es_path = results_dir / 'event_study_ff5_results.csv'
if es_path.exists():
    es = pd.read_csv(es_path)
    # Format for publication
    es_pub = es[['ticker', 'event_label', 'event_date', 'car', 't_stat', 'significant_5pct']].copy()
    es_pub['car'] = es_pub['car'].apply(lambda x: f'{x:+.4f}')
    es_pub['t_stat'] = es_pub['t_stat'].apply(lambda x: f'{x:+.2f}')
    es_pub.to_csv(figures_dir / 'table_event_study.csv', index=False)
    print('Table: Event Study Results')
    print(es_pub.to_string(index=False))
else:
    print('event_study_ff5_results.csv not found — run 05_event_study.ipynb first.')

# Also check for earnings-calibration cross data
ec_path = results_dir / 'earnings_calibration_context.csv'
if ec_path.exists():
    ec = pd.read_csv(ec_path)
    ec.to_csv(figures_dir / 'table_earnings_calibration.csv', index=False)
    print(f'\nTable: Earnings-Calibration Context ({len(ec)} events)')
    print(ec.to_string(index=False))

## 5. Output Manifest

In [ ]:
# List all generated outputs
tables = sorted(figures_dir.glob('table_*.csv'))
figures = sorted(figures_dir.glob('*.png'))

print(f'=== Thesis Output Manifest ===')
print(f'\nTables ({len(tables)}):')
for t in tables:
    print(f'  {t.name}')

print(f'\nFigures ({len(figures)}):')
for f in figures:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name} ({size_kb:.0f} KB)')

# Create manifest CSV
manifest = []
for t in tables:
    manifest.append({'type': 'table', 'file': t.name, 'size_kb': round(t.stat().st_size/1024, 1)})
for f in figures:
    manifest.append({'type': 'figure', 'file': f.name, 'size_kb': round(f.stat().st_size/1024, 1)})
pd.DataFrame(manifest).to_csv(figures_dir / 'output_manifest.csv', index=False)
print(f'\nManifest saved to output_manifest.csv')